## Celem tego notebooka jest scrapowanie strony [lubimy czytać](https://lubimyczytac.pl/) w celu pobrania danych dotyczących rankingu "[top100](https://lubimyczytac.pl/top100)" książek ze wszystkich dostępnych na stronie okresów czasu i wstępna analiza

### Import bibliotek

In [108]:
from bs4 import BeautifulSoup
import requests     
import time
import random
import pandas as pd

### Konfiguracja

In [53]:
# URL strony z listą książek
BASE_URL = "https://lubimyczytac.pl/top100"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:151.0) Gecko/20100101 Firefox/151.0"
}

# Opóźnienie między żądaniami (w sekundach)
SLEEP_MIN = 1
SLEEP_MAX = 3
#time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX)) # do użycia później, między żądaniami

### Test requesta

In [58]:
request = requests.get(BASE_URL, headers=headers)
print(request.status_code)
soup = BeautifulSoup(request.text, "html.parser")

200


### Zapis surowego HTML do pliku (do testów i szukania odpowiednich selektorów)

In [112]:
with open("data/html/raw_html.txt", "w", encoding="utf8") as f:
    f.write(soup.prettify())

### Analiza html - wczytanie dat potrzebnych do przechodzenia po archiwum

In [59]:
months = soup.find_all("div", class_="filtr__itemTitle")

month_map = {
    "Styczeń": 1,
    "Luty": 2,
    "Marzec": 3,
    "Kwiecień": 4,
    "Maj": 5,
    "Czerwiec": 6,
    "Lipiec": 7,
    "Sierpień": 8,
    "Wrzesień": 9,
    "Październik": 10,
    "Listopad": 11,
    "Grudzień": 12
}

archives = []

for month in months:
    text = month.get_text(strip=True)

    month_name, year = text.split()

    archives.append({
        "year": int(year),
        "month": month_map[month_name]
    })

print(len(archives))

12


### Analiza html - parsowanie strony i wyciąganie istotnych danych

In [113]:
#all_books_eng = []
all_books_pl = []

for archive in archives:    
    for page in range(1, 14):
        # Opóźnienie między żądaniami, aby nie przeciążać serwera i uniknąć blokady
        time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))

        url = f"https://lubimyczytac.pl/top100?page={page}&listId=listTop100&month={archive['month']}&year={archive['year']}&paginatorType=Standard"
        response = requests.get(url, headers=headers)

        if response.status_code != 200:
            print(f"Nie można pobrać strony {page}, status code: {response.status_code}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        print(f"Pobrano stronę {page} dla {archive['month']}-{archive['year']}")

        ranking = soup.find("div", class_="list-standard--listTop100")
        book_cards = ranking.find_all("div", class_="book-card")
        print(f"Znaleziono {len(book_cards)} książek na stronie")
        for book in book_cards:
            try:  
                title = book.find("a", class_="book-card__title").get_text(strip=True)
                author = book.find("div", class_="book-card__author").get_text(strip=True)
                rating = book.find("span", class_="rating__avarage").get_text(strip=True)
                rating_counts = int(
                    book.find("span", class_="rating__text").contents[-1]
                    .get_text(strip=True)
                    .replace("ocen", "")
                    .strip()
                )
                ranking_number = book.find("span", class_="book-card__label").contents[0].get_text(strip=True)
                ranking_change = book.find("span", class_="book-card__label").contents[1].get_text(strip=True)
                month_year = f"{archive['month']:02d}-{archive['year']}"
                readers_count = book.find("span", class_="book-card__readers").find("span").get_text(strip=True)
                reviews_count = book.find("span", class_="book-card__reviews").find("span").get_text(strip=True)
                
                all_books_pl.append({
                    "okres": month_year,
                    "tytuł": title,
                    "autor": author,
                    "pozycja_w_rankingu": ranking_number,
                    "zmiana_w_rankingu": ranking_change,
                    "ocena": rating,
                    "liczba_ocen": rating_counts,
                    "liczba_czytelników": readers_count,
                    "liczba_recenzji": reviews_count
                })
                
                #all_books_eng.append({
                #    "month_year": month_year,
                #    "title": title,
                #    "author": author,
                #    "ranking_number": ranking_number,
                #    "ranking_change": ranking_change,
                #    "rating": rating,
                #    "rating_counts": rating_counts,
                #    "readers_count": readers_count,
                #    "reviews_count": reviews_count
                #})
            except Exception as e:
                print(f"Błąd podczas przetwarzania książki: {e}")
                continue        


Pobrano stronę 1 dla 4-2026
Znaleziono 10 książek na stronie
Pobrano stronę 2 dla 4-2026
Znaleziono 10 książek na stronie
Pobrano stronę 3 dla 4-2026
Znaleziono 10 książek na stronie
Pobrano stronę 4 dla 4-2026
Znaleziono 10 książek na stronie
Pobrano stronę 5 dla 4-2026
Znaleziono 10 książek na stronie
Pobrano stronę 6 dla 4-2026
Znaleziono 10 książek na stronie
Pobrano stronę 7 dla 4-2026
Znaleziono 10 książek na stronie
Pobrano stronę 8 dla 4-2026
Znaleziono 10 książek na stronie
Pobrano stronę 9 dla 4-2026
Znaleziono 10 książek na stronie
Pobrano stronę 10 dla 4-2026
Znaleziono 10 książek na stronie
Pobrano stronę 11 dla 4-2026
Znaleziono 10 książek na stronie
Pobrano stronę 12 dla 4-2026
Znaleziono 2 książek na stronie
Nie można pobrać strony 13, status code: 404
Pobrano stronę 1 dla 3-2026
Znaleziono 10 książek na stronie
Pobrano stronę 2 dla 3-2026
Znaleziono 10 książek na stronie
Pobrano stronę 3 dla 3-2026
Znaleziono 10 książek na stronie
Pobrano stronę 4 dla 3-2026
Znaleziono

### Zapis danych do csv

In [116]:
df_pl =pd.DataFrame(all_books_pl)

df_pl.to_csv("data/csv/lubimy_czytac_top100_pl.csv", index=False, encoding="utf-8-sig")
df_pl.head()

,okres,tytuł,autor,pozycja_w_rankingu,zmiana_w_rankingu,ocena,liczba_ocen,liczba_czytelników,liczba_recenzji
0,04-2026,Szczęście czy fart?,"Trias de Bes Fernando,Alex Rovira Celma",1,⬈,"7,4",359,1395,42
1,04-2026,Projekt Hail Mary,Andy Weir,2,⬊,"8,1",7450,14356,1269
2,04-2026,Święto Karkonoszy,Sławek Gortych,3,★,"7,7",681,2401,147
3,04-2026,Cały ten błękit,Mélissa Da Costa,4,⬊,"7,8",2948,8801,560
4,04-2026,Gołoborze,Maciej Siembieda,5,⬊,"8,3",4219,9777,872


### W celu większej przejrzystości pre-processing danych i analiza zostaną przeprowadzone w osobnym notebooku.

In [117]:
df_pl.info()

<class 'pandas.DataFrame'>
RangeIndex: 1468 entries, 0 to 1467
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   okres               1468 non-null   str  
 1   tytuł               1468 non-null   str  
 2   autor               1468 non-null   str  
 3   pozycja_w_rankingu  1468 non-null   str  
 4   zmiana_w_rankingu   1468 non-null   str  
 5   ocena               1468 non-null   str  
 6   liczba_ocen         1468 non-null   int64
 7   liczba_czytelników  1468 non-null   str  
 8   liczba_recenzji     1468 non-null   str  
dtypes: int64(1), str(8)
memory usage: 103.3 KB
